# 09 Lagra

Vi har sparat data i varje steg. I den här notebooken ser vi över hur vi lagrar den, så att den är lätt att hitta, fråga och dela utan att något går förlorat på vägen.

Vi tittar på tre saker.

1. Hur mapparna speglar stegen i arbetet.
2. Varför formatet spelar roll, med CSV och Parquet som exempel.
3. Hur vi kan ställa frågor till datan med SQL i en databas, DuckDB.

In [ ]:
import json
from pathlib import Path

import duckdb
import pandas as pd

DATA = Path("..") / "data"

## Mapparna speglar stegen

Varje steg i arbetet har sin egen mapp. Det gör det lätt att se var en fil kommer ifrån och vilka filer vi kan kasta och skapa igen.

- `radata` är svaren från källorna som de kom. Dem ändrar vi aldrig.
- `tvattad` är en tabell per källa, skapad ur rådatan i notebook 06.
- `harmoniserad` är den sammanslagna tabellen från notebook 07.

`rglob("*")` letar upp alla filer i mappen och dess undermappar. `stat().st_size` är filens storlek i byte, som vi delar med 1024 för att få kilobyte.

In [ ]:
for fil in sorted(DATA.rglob("*")):
    if fil.is_file() and fil.name != ".gitkeep":
        print(f"{fil.relative_to(DATA)}  {fil.stat().st_size / 1024:.0f} kB")

`:.0f` skriver talet utan decimaler.

Allt utom rådatan kan vi skapa igen genom att köra notebook 06 och 07. Rådatan kan vi inte skapa igen, eftersom källorna ändras över tid. Det är den som behöver säkerhetskopieras om vi vill kunna upprepa en analys.

## Formatet spelar roll

Vi läser in den sammanslagna tabellen och sparar den också som CSV, ett vanligt textformat där värdena skiljs åt med kommatecken.

In [ ]:
webbplatser = pd.read_parquet(DATA / "harmoniserad" / "webbplatser.parquet")

csv_fil = DATA / "harmoniserad" / "webbplatser.csv"
webbplatser.to_csv(csv_fil, index=False)

fran_csv = pd.read_csv(csv_fil)

Nu jämför vi några kolumner i originalet med samma kolumner när de har gått via CSV.

In [ ]:
kolumner = ["namn", "kommunkod", "lanskod", "befolkning", "period_slut"]

pd.DataFrame({
    "original": webbplatser[kolumner].dtypes.astype(str),
    "via csv": fran_csv[kolumner].dtypes.astype(str),
})

In [ ]:
pd.DataFrame({
    "original": webbplatser.loc[0, kolumner],
    "via csv": fran_csv.loc[0, kolumner],
})

CSV sparar bara text. När pandas läser filen gissar den vilken typ varje kolumn har. Kommunkoden `1440` ser ut som ett tal och blir ett decimaltal, eftersom kolumnen saknar värde för regionerna. En kommunkod som `0180` för Stockholm blir då `180.0`. Länskoden tappar sin nolla på samma sätt, och datumen blir text.

Vi kan tala om för pandas vilka typer kolumnerna har med `dtype` och `parse_dates`. Då måste vi veta det, och komma ihåg det varje gång filen läses.

In [ ]:
fran_csv = pd.read_csv(
    csv_fil,
    dtype={"kommunkod": "str", "kolada_id": "str", "lanskod": "str"},
    parse_dates=["befolkning_datum", "period_start", "period_slut"],
)

fran_csv.loc[fran_csv["namn"] == "Stockholms kommun", ["namn", "kommunkod", "lanskod"]]

Parquet sparar datatypen tillsammans med värdena, så där behövs inga sådana uppgifter. Parquet packar också datan, så filen blir mindre.

In [ ]:
for fil in [DATA / "harmoniserad" / "webbplatser.parquet", csv_fil]:
    print(fil.name, round(fil.stat().st_size / 1024), "kB")

CSV har ändå en fördel. Filen går att öppna i ett kalkylprogram och läsa i vilken textredigerare som helst. Därför är CSV bra när vi delar data med någon som inte programmerar. Parquet är bättre för datan vi själva arbetar vidare med.

Svenska kalkylprogram använder ofta semikolon mellan kolumner och kommatecken som decimaltecken. Med `sep` och `decimal` skapar vi en fil som öppnas rätt där.

In [ ]:
delning = DATA / "harmoniserad" / "webbplatser-kalkylark.csv"
webbplatser.to_csv(delning, index=False, sep=";", decimal=",", encoding="utf-8-sig")

delning.read_text(encoding="utf-8-sig").splitlines()[1][:120]

`utf-8-sig` lägger till ett osynligt tecken först i filen som hjälper Excel att visa å, ä och ö rätt.

## Fråga datan med SQL

SQL är ett språk för att ställa frågor till tabeller, som finns i nästan alla databaser. DuckDB är en databas som ligger i en enda fil och inte kräver någon server. Den kan läsa Parquet direkt.

In [ ]:
databas = duckdb.connect(str(DATA / "webbanalys.duckdb"))

databas.sql(f'''
    CREATE OR REPLACE TABLE webbplatser AS
    SELECT * FROM read_parquet('{DATA / "harmoniserad" / "webbplatser.parquet"}')
''')

databas.sql("SELECT COUNT(*) AS antal FROM webbplatser").df()

`CREATE OR REPLACE TABLE` skapar tabellen, eller ersätter den om den redan finns. `.df()` gör om svaret till en pandas-tabell.

En fråga i SQL säger vilka kolumner vi vill ha med `SELECT`, från vilken tabell med `FROM`, och vilka rader med `WHERE`. Här är kommunerna i Gävleborgs län med lägst LCP först.

In [ ]:
databas.sql('''
    SELECT namn, befolkning, lcp_p75_ms
    FROM webbplatser
    WHERE lan = 'Gävleborgs län' AND typ = 'kommun'
    ORDER BY lcp_p75_ms
''').df()

Med `GROUP BY` räknar vi per grupp. Här är antalet kommuner och medianen för LCP per län. `median` är det mittersta värdet när de sorteras, och påverkas inte av enstaka mycket höga värden på samma sätt som medelvärdet.

In [ ]:
databas.sql('''
    SELECT lan, COUNT(*) AS kommuner, median(lcp_p75_ms) AS lcp_median_ms
    FROM webbplatser
    WHERE typ = 'kommun'
    GROUP BY lan
    ORDER BY lcp_median_ms
''').df()

Samma sak går att göra i pandas med `groupby`. Vilket vi väljer är en smaksak. SQL är bra att kunna eftersom det fungerar likadant i de flesta databaser och verktyg.

Vi stänger förbindelsen när vi är klara, så att filen inte är låst.

In [ ]:
databas.close()

## Beskriv datan

Den som får vår data behöver veta var den kommer ifrån, när den hämtades och vilka villkor som gäller. Vi sparar det i en liten fil bredvid datan.

Licenserna avgör hur resultatet får delas. Webperfs data är licensierad under CC BY-SA 4.0. Det betyder att vi anger Webperf som källa och att det vi bygger på datan delas under samma licens. CrUX är licensierad under CC BY 4.0 och kräver att vi anger källan. Wikidata är CC0 och fri att använda. Koladas villkor finns på [kolada.se](https://www.kolada.se).

In [ ]:
beskrivning = {
    "fil": "webbplatser.parquet",
    "beskrivning": "En rad per webbplats för Sveriges kommuner och regioner, med betyg, fältdata, befolkning och bredband.",
    "licens": "CC BY-SA 4.0",
    "kallor": [
        {"namn": "Webperf", "url": "https://api.webperf.se", "licens": "CC BY-SA 4.0",
         "hamtad": str(webbplatser["hamtad"].min())},
        {"namn": "Wikidata", "url": "https://www.wikidata.org", "licens": "CC0",
         "hamtad": str(webbplatser["hamtad_wikidata"].min())},
        {"namn": "Chrome UX Report", "url": "https://developer.chrome.com/docs/crux", "licens": "CC BY 4.0",
         "hamtad": str(webbplatser["hamtad_crux"].min())},
        {"namn": "Kolada", "url": "https://www.kolada.se", "licens": "se kolada.se",
         "ar": int(webbplatser["kolada_ar"].iloc[0])},
    ],
}

fil = DATA / "harmoniserad" / "om-datan.json"
fil.write_text(json.dumps(beskrivning, ensure_ascii=False, indent=2), encoding="utf-8")
print(fil.read_text(encoding="utf-8"))

## Det här vet vi nu

- Mappar för rådata, tvättad och harmoniserad data visar var varje fil kommer ifrån.
- Rådatan är det enda som inte går att skapa igen.
- CSV tappar datatyper, till exempel nollan först i en kommunkod. Parquet behåller dem.
- CSV är ändå bra för att dela med den som använder kalkylprogram.
- DuckDB låter oss fråga datan med SQL utan en server.
- Datan behöver en beskrivning med källor, tider och licenser.

I nästa notebook visualiserar vi datan och prövar vad den kan och inte kan säga.